# Week 11: Decision-Making & Planning in Agents
## Complete Teaching Notebook — PPT Flow Reference

**Course:** Professional Certificate Programme in Agentic AI and Applications
**Duration:** 6–7 hours across 2 days

---

### How to use this notebook
Each section maps to a **PPT Block** with full instructor scripts.
Look for the 🎙 **INSTRUCTOR SCRIPT** boxes — these are your talking points.
Look for 🌍 **REAL-WORLD EXAMPLE** boxes — use these to anchor theory in practice.
Look for ❓ **Ask the class** prompts — pause here and wait for answers.

| PPT Block | Notebook Section | API Key? |
|-----------|-----------------|----------|
| Block 2 | Part 1 — Goal Execution Pipeline | **No** |
| Block 3 | Part 2 — Hierarchical Planning (Vacation) | Yes |
| Block 4 | Part 3 — Multi-Agent Coordination (Customer Support) | Yes |
| Block 5 | Part 4 — Healthcare Triage (All 4 Planning Modes) | Yes |
| Block 7 | Part 5 — Reinforcement Loop | Yes |
| Block 8 | Part 6 — Hybrid ReAct Agent | Yes |
| Capstone | Part 7 — RL + Crew Integration | Yes |


---
## Setup — Run this cell first every session

```bash
pip install crewai==0.28.8 langchain-openai python-dotenv
```

Create a `.env` file in this folder with:
```
OPENAI_API_KEY=sk-...your-key...
```


In [ ]:
# ── Install (uncomment if not installed) ─────────────────────
# !pip install crewai==0.28.8 langchain-openai python-dotenv

import warnings
warnings.filterwarnings('ignore')

import os
from dotenv import load_dotenv
load_dotenv()   # reads OPENAI_API_KEY from .env

from langchain_openai import ChatOpenAI

# gpt-4o-mini: fast, cheap, good enough for all demos in this course
# temperature=0 → deterministic output → reproducible results for teaching
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

api_ok = bool(os.getenv("OPENAI_API_KEY"))
print("=" * 50)
print("  Week 11 — Teaching Environment Ready")
print("=" * 50)
print(f"  Model      : gpt-4o-mini (temperature=0)")
print(f"  API Key    : {'Loaded ✓' if api_ok else 'NOT FOUND — add to .env ✗'}")
print(f"  Pure-Python sections (Parts 1 & 5) work without API key.")
print("=" * 50)


---
# Part 1 — Goal Execution Pipeline
### PPT: Block 2 — Slides 8–10 | ⏱ 20 minutes | No API key needed

**The concept in one sentence:**
A real agent runs every user query through a 5-stage pipeline before acting —
just like a surgeon doesn't pick up a scalpel the moment the patient walks in.

```
User Query → Interpretation → Decomposition → Planning → Execution → Reflection
```


In [ ]:
# ═══════════════════════════════════════════════════════════
#  PART 1 — Goal Execution Pipeline (pure Python)
#  Run this without any API key — pure simulation.
#  Each method maps to a real agent behaviour.
# ═══════════════════════════════════════════════════════════

class GoalExecutionPipeline:
    '''
    5-stage pipeline that mirrors what a real agent does internally.

    In production each stage is an LLM call.
    Here we simulate with keyword logic so you can see the structure
    without waiting for API responses.
    '''

    def __init__(self, name: str):
        self.name = name
        # history = the Reflection stage output that grows over time
        # In a real agent this feeds the reinforcement loop
        self.history = []

    # ── Stage 1: Interpretation ──────────────────────────────
    # Real agent: LLM call — "What does the user ACTUALLY want?"
    # Strips ambiguity, infers intent beyond the literal words.
    def interpret(self, raw_query: str) -> str:
        intent_map = {
            "book":      "Reserve and confirm the requested resource with all constraints",
            "find":      "Search across available sources and return ranked results",
            "summarise": "Condense provided content into the top 3-5 key points",
            "plan":      "Create a structured, ordered, multi-step action plan",
            "analyse":   "Evaluate the subject and produce a structured findings report",
            "fix":       "Identify the root cause and implement the minimum viable fix",
        }
        for keyword, intent in intent_map.items():
            if keyword in raw_query.lower():
                return intent
        return f"Fulfil the stated request: '{raw_query}'"

    # ── Stage 2: Decomposition ───────────────────────────────
    # Real agent: LLM call — breaks intent into ordered sub-tasks with dependencies.
    # Teaching point: this is where 1 goal becomes many tasks.
    # In CrewAI: this is your Task list inside the Crew object.
    def decompose(self, intent: str) -> list:
        return [
            f"[T1] Gather all information required for: {intent}",
            f"[T2] Validate and filter gathered data — remove noise and duplicates",
            f"[T3] Execute the core action: {intent}",
            f"[T4] Verify outcome matches the original intent — quality gate",
        ]

    # ── Stage 3: Planning ────────────────────────────────────
    # Real agent: assigns tools to tasks, resolves dependencies, handles failure modes.
    # In CrewAI: Task(agent=X) is the planning step — you assign who does what.
    def plan(self, sub_tasks: list) -> dict:
        return {
            "execution_order": sub_tasks,
            "tool_assignments": {
                sub_tasks[0]: "web_search / database_query",
                sub_tasks[1]: "data_filter / deduplication",
                sub_tasks[2]: "api_call / LLM_generation",
                sub_tasks[3]: "human_review / automated_verifier",
            },
            # CRITICAL: failure mode is defined at planning time, not after failure
            "failure_mode": (
                "If T1 returns empty → HALT and ask user for clarification. "
                "Do NOT proceed to T3 with empty data."
            ),
        }

    # ── Stage 4: Execution ───────────────────────────────────
    # Real agent: tool calls, API calls, CrewAI crew.kickoff().
    # Simulation: deterministic result for first 3 tasks, flag on task 4.
    def execute(self, plan: dict) -> dict:
        results = {}
        for i, (task, tool) in enumerate(plan["tool_assignments"].items()):
            # Task 4 (verification) often flags issues — realistic
            status  = "SUCCESS" if i < 3 else "NEEDS HUMAN REVIEW"
            results[task] = {"status": status, "tool_used": tool}
        return results

    # ── Stage 5: Reflection ──────────────────────────────────
    # This is what separates automation from intelligence.
    # Real agent: LLM scores output quality; updates learning_history.
    # Without reflection: the agent makes the same mistake every run.
    def reflect(self, results: dict, raw_query: str) -> dict:
        successes = sum(1 for r in results.values() if r["status"] == "SUCCESS")
        total     = len(results)
        rate      = successes / total

        reflection = {
            "original_query": raw_query,
            "success_rate":   f"{rate:.0%}",
            "steps_passed":   successes,
            "steps_total":    total,
            "lesson": (
                "All steps succeeded — no strategy change needed."
                if rate == 1.0 else
                "Step 4 required human review — add a dedicated verification agent next run."
            ),
        }
        self.history.append(reflection)
        return reflection

    # ── Full pipeline runner ─────────────────────────────────
    def run(self, raw_query: str):
        print(f"
{'═'*62}")
        print(f"  Agent: {self.name}  |  Query: {raw_query}")
        print(f"{'═'*62}")

        print("
  STAGE 1 — Interpretation")
        intent = self.interpret(raw_query)
        print(f"  → Raw query   : '{raw_query}'")
        print(f"  → Interpreted : '{intent}'")

        print("
  STAGE 2 — Decomposition (1 goal → 4 sub-tasks)")
        sub_tasks = self.decompose(intent)
        for t in sub_tasks:
            print(f"    {t}")

        print("
  STAGE 3 — Planning (assign tools + define failure mode)")
        plan = self.plan(sub_tasks)
        for task, tool in plan["tool_assignments"].items():
            print(f"    {task[:55]:<55} → {tool}")
        print(f"  → Failure mode : {plan['failure_mode']}")

        print("
  STAGE 4 — Execution")
        results = self.execute(plan)
        for task, outcome in results.items():
            icon = "✓" if outcome["status"] == "SUCCESS" else "⚠"
            print(f"  {icon}  {task[:52]:<52} [{outcome['status']}]")

        print("
  STAGE 5 — Reflection (log outcome, extract lesson)")
        ref = self.reflect(results, raw_query)
        print(f"  → Success rate : {ref['success_rate']}")
        print(f"  → Lesson       : {ref['lesson']}")
        print(f"{'─'*62}")
        return ref


In [ ]:
# ── Run the pipeline on 3 different query types ──────────────
# Same pipeline — different industries — shows generality

pipeline = GoalExecutionPipeline(name="GeneralistAgent")

print("\n### QUERY 1 — Operations (booking)")
pipeline.run("Book a meeting room for 10 people next Tuesday at 2pm")

print("\n### QUERY 2 — Research (competitive intelligence)")
pipeline.run("Find the top 5 competitors of our product in the European healthcare market")

print("\n### QUERY 3 — Engineering (debugging)")
pipeline.run("Fix the memory leak in the payments microservice")


In [ ]:
# ── After 3 runs — show what the agent learned ────────────────
# history grows with each run — this IS the reinforcement loop foundation

print("Agent learning history across all runs:")
print(f"{'─'*60}")
for i, entry in enumerate(pipeline.history, 1):
    print(f"  Run {i}: {entry['original_query'][:45]:<45}")
    print(f"         Success: {entry['success_rate']} | Lesson: {entry['lesson']}")
    print()

print(f"Total runs logged: {len(pipeline.history)}")
print("This history will feed the reinforcement loop in Part 5.")


---
### ★ Quick Check (Block 2 Quiz)

Answer these before moving on — no peeking at the code:

1. **What does Stage 1 (Interpretation) prevent?** *(Hint: think about what a bad employee does immediately)*
2. **In CrewAI, which object represents Stage 3 (Planning)?**
3. **Why does history grow with each pipeline run?**
4. **What would happen if you removed Stage 5 (Reflection) from production?**

---


---
# Part 2 — Hierarchical Planning
### PPT: Block 3 — Slides 13–17 | ⏱ 20 minutes | Requires API key

**The concept:** Top-down decomposition.
One high-level goal → layers of sub-goals → each handled by a specialist.

**Why 5 agents instead of 1?**
An LLM has finite attention. When you give one agent 5 jobs,
it does each of them worse. Specialist agents with narrow jobs
outperform generalist agents with wide ones — every time.

```
planner → researcher → budgeter → itinerary_maker → coordinator
```


In [ ]:
# ═══════════════════════════════════════════════════════════
#  PART 2 — Hierarchical Planning: Family Vacation to Italy
# ═══════════════════════════════════════════════════════════

from crewai import Agent, Task, Crew

# ── Before defining agents, understand the HIERARCHY ────────
#
#   planner (Level 1)          — strategic decomposition
#       ↓
#   researcher, budgeter,      — specialist execution (Level 2)
#   itinerary_maker
#       ↓
#   coordinator (Integration)  — conflict resolution, final assembly
#
# Teaching point: the coordinator is NOT the same as a manager.
# Its job is to RESOLVE CONFLICTS between the other agents.
# e.g. budgeter says "mid-range hotel in Florence",
#      researcher recommends a neighbourhood that has no mid-range hotels.
# The coordinator catches that contradiction.

planner = Agent(
    role="High-Level Trip Planner",
    goal="Decompose the vacation goal into a clear hierarchy: pillars → sub-tasks → success criteria.",
    backstory=(
        "You are a senior travel architect. You break large, vague goals into "
        "structured hierarchies with explicit success criteria at every level. "
        "You do NOT book anything or research prices — that is not your job. "
        "Your job is the STRUCTURE of the plan."
    ),
    llm=llm,
    verbose=False  # Change to verbose=True during live demo to see agent reasoning
)

researcher = Agent(
    role="Travel Researcher",
    goal="Find 2 practical options each for flights, accommodation, and local transport.",
    backstory=(
        "You are a travel research specialist. Given a destination and dates, "
        "you produce concise option comparisons — never lengthy essays. "
        "You flag anything needing external booking with 'ACTION REQUIRED'."
    ),
    llm=llm,
    verbose=False
)

budgeter = Agent(
    role="Family Budget Analyst",
    goal="Estimate total trip cost by category and provide 2 concrete cost-saving suggestions.",
    backstory=(
        "You are a family travel financial planner. You produce realistic cost tables "
        "and identify where families typically overspend. "
        "You never invent exact prices — you use ranges (e.g. 150–200 EUR/night)."
    ),
    llm=llm,
    verbose=False
)

itinerary_maker = Agent(
    role="Family Itinerary Creator",
    goal="Build a realistic day-by-day itinerary with travel time, rest buffers, and child-friendly notes.",
    backstory=(
        "You design itineraries for families with young children. "
        "You know that a 5-year-old cannot walk 15km per day. "
        "You build in rest time, nap time, and short-distance travel between sites. "
        "Each day includes Morning / Afternoon / Evening items."
    ),
    llm=llm,
    verbose=False
)

# ── The coordinator: the integration layer ───────────────────
# Without this agent, outputs from researcher and itinerary_maker
# can contradict each other. The coordinator reads ALL previous outputs
# and resolves inconsistencies before the final plan is delivered.
coordinator = Agent(
    role="Trip Coordinator & Quality Checker",
    goal=(
        "Assemble a single, conflict-free final travel plan. "
        "Ensure hotel cities match flight arrival cities. "
        "Ensure budget aligns with researched options. "
        "Flag any contradiction found between agent outputs."
    ),
    backstory=(
        "You are the senior travel consultant who reads every specialist's work "
        "and catches errors before they reach the client. "
        "You have caught hundreds of cases where the hotel was booked in the wrong city. "
        "You are the last human (or AI) review before the plan goes live."
    ),
    llm=llm,
    verbose=False
)


In [ ]:
# ── Task definitions — the expected_output is the inter-agent contract ──

# Task 1: Planner builds the hierarchy
# → researcher uses this to know which cities to research
decompose_task = Task(
    description=(
        "Decompose this vacation goal into a structured plan:\n"
        "  Goal        : {vacation_goal}\n"
        "  Dates       : {dates}\n"
        "  Family size : {family_size}\n"
        "  Preferences : {preferences}\n\n"
        "Produce:\n"
        "  Level 1 : 5 planning pillars (flights, lodging, local transport, activities, budget)\n"
        "  Level 2 : For each pillar, list 3 concrete sub-tasks\n"
        "  Level 3 : For the highest-priority sub-task per pillar, state the success criterion"
    ),
    expected_output="Structured hierarchy as plain text: pillar → sub-tasks → success criteria",
    agent=planner
)

# Task 2: Researcher finds options
# expected_output format tells knowledge_agent exactly what to produce
# so the next agent (budgeter) can parse it reliably
research_task = Task(
    description=(
        "Research travel options for: {vacation_goal}  |  {dates}  |  {family_size}\n\n"
        "Return EXACTLY 2 options each for:\n"
        "  Flights      : price tier (budget/mid/premium) + 1-line rationale\n"
        "  Accommodation: hotel vs apartment, neighbourhood name, proximity to sights\n"
        "  Local transport: rail pass vs rental car — pros/cons for family with children\n\n"
        "Mark any option requiring external booking: 'ACTION REQUIRED: [reason]'"
    ),
    expected_output="2 options per category (flights / accommodation / transport), clearly labelled",
    agent=researcher
)

# Task 3: Budgeter creates cost table
# Uses researcher's output (previous task result) as context
budget_task = Task(
    description=(
        "Create a cost estimate for: {vacation_goal}  |  {dates}  |  {family_size}\n"
        "Preferences: {preferences}\n\n"
        "Produce a cost table with RANGES (not exact figures):\n"
        "  Flights (return, all passengers) | Accommodation per night | Local transport (total) |"
        "  Food per day | Activities (total) | Contingency (10%)\n\n"
        "End with: TOTAL ESTIMATED BUDGET and 2 specific money-saving suggestions."
    ),
    expected_output="Cost table with ranges, total budget, and 2 saving suggestions",
    agent=budgeter
)

# Task 4: Itinerary maker builds day-by-day schedule
# Key: child-friendly pace is a domain constraint baked into the task description
itinerary_task = Task(
    description=(
        "Build a day-by-day itinerary for: {vacation_goal}  |  {family_size}\n"
        "Preferences: {preferences}\n\n"
        "For each day list:\n"
        "  - Morning / Afternoon / Evening items (max 2 sites per half-day — family pace)\n"
        "  - Estimated travel time between stops\n"
        "  - Child-friendly note (age {family_size}: what works for under-10s)\n"
        "  - Any booking required in advance\n\n"
        "Include at least ONE rest afternoon in the 7-day schedule — families need downtime."
    ),
    expected_output="Full day-by-day itinerary with timing notes and child-friendly annotations",
    agent=itinerary_maker
)

# Task 5: Coordinator integrates everything
# This agent reads ALL previous task outputs as context and resolves conflicts
coordinate_task = Task(
    description=(
        "You have received outputs from 4 specialist agents for: {vacation_goal}\n\n"
        "Your job:\n"
        "  1. Cross-check that hotel cities MATCH the cities in the itinerary\n"
        "  2. Cross-check that the budget covers the researched accommodation options\n"
        "  3. Confirm itinerary travel times are realistic (e.g. Rome to Florence: 1.5hr by train)\n\n"
        "Then produce the FINAL travel plan:\n"
        "  - Executive summary (2 sentences)\n"
        "  - Booking checklist with TIMING (what to book now vs 2 weeks out vs 1 week out)\n"
        "  - 3 risk notes and contingencies\n"
        "  - Final total estimated cost range\n\n"
        "If you find any contradiction between agent outputs, state it explicitly as:\n"
        "  CONFLICT FOUND: [description] → RESOLUTION: [your fix]"
    ),
    expected_output="Final plan: summary + booking checklist + risk notes + total cost + any conflicts resolved",
    agent=coordinator
)


In [ ]:
# ── Build and run the vacation crew ──────────────────────────
#
# verbose=False keeps the output clean for class discussion.
# During live demo: change to verbose=2 to watch each agent's reasoning.
#
# The Crew object = the Strategic Foundation (Layer 1 of the 3-layer model).
# It defines WHAT happens. The agents decide HOW.

vacation_crew = Crew(
    agents=[planner, researcher, budgeter, itinerary_maker, coordinator],
    tasks=[decompose_task, research_task, budget_task, itinerary_task, coordinate_task],
    verbose=False
)

inputs = {
    "vacation_goal": "Plan a 7-day family vacation to Italy: Rome → Florence → Venice",
    "dates":         "15 June 2025 to 22 June 2025",
    "family_size":   "2 adults, 2 children aged 8 and 5",
    "preferences":   "Moderate budget, child-friendly activities, central hotels, no overnight trains",
}

print("Running 5-agent hierarchical crew...")
print("Each agent calls gpt-4o-mini once. 5 calls total, ~60-90 seconds.\n")

result = vacation_crew.kickoff(inputs=inputs)

print("\n" + "═"*62)
print("  FINAL VACATION PLAN (from coordinator agent)")
print("═"*62)
print(result)


In [ ]:
# ── TEACHING MOMENT: What happens with only 1 agent? ─────────
# This demonstrates WHY decomposition matters.
# Run this and compare the output to the 5-agent version above.

single_agent = Agent(
    role="All-in-One Vacation Planner",
    goal="Plan a complete 7-day family vacation to Italy: research, budget, itinerary, coordination.",
    backstory="You handle every aspect of trip planning: research, budgeting, scheduling, and coordination.",
    llm=llm,
    verbose=False
)

single_task = Task(
    description=(
        "Plan a complete 7-day family vacation to Italy (Rome, Florence, Venice)\n"
        "for 2 adults and 2 children aged 8 and 5.\n"
        "Dates: 15 June 2025 to 22 June 2025\n"
        "Budget: moderate. Preferences: child-friendly, central hotels, no overnight trains.\n\n"
        "Produce: hotel recommendations, flights, budget, full 7-day itinerary, booking checklist."
    ),
    expected_output="Complete vacation plan including all of: hotels, flights, budget, itinerary",
    agent=single_agent
)

single_crew = Crew(agents=[single_agent], tasks=[single_task], verbose=False)

print("Running SINGLE agent for the same vacation goal...")
print("(No decomposition, no specialisation, all in one LLM call)\n")
single_result = single_crew.kickoff(inputs={})
print("\n" + "═"*62)
print("  SINGLE-AGENT OUTPUT (compare to 5-agent output above)")
print("═"*62)
print(single_result)


---
### ★ Exercise — Change the Domain

Replace the `inputs` dict with a business trip scenario:
```python
inputs = {
    "vacation_goal": "Plan a 3-day executive business trip to Singapore for a product launch",
    "dates":         "1 September 2025 to 3 September 2025",
    "family_size":   "1 executive (senior VP)",
    "preferences":   "Premium budget, hotel near Marina Bay Sands Convention Centre, dinner reservations required",
}
```

Notice how every agent adapts its output without any code changes.
That is **goal-driven** behaviour — the agent reads its task description and adapts.

---


---
# Part 3 — Multi-Agent Coordination & Task Chaining
### PPT: Block 4 — Slides 19–25 | ⏱ 15 minutes | Requires API key

**The concept:** Task chaining — Agent N's output IS Agent N+1's context.
The `expected_output` field is the **contract** between agents.

**Pipeline:**
```
issue_classifier → knowledge_agent → solution_agent → escalation_agent
```


In [ ]:
# ═══════════════════════════════════════════════════════════
#  PART 3 — Multi-Agent Coordination: Customer Support
# ═══════════════════════════════════════════════════════════

from crewai import Agent, Task, Crew

# ── 4 specialist agents — each owns ONE step of the pipeline ─

# Agent 1: Classifier — produces structured JSON
# The quality of EVERY downstream agent depends on this agent's output format.
# Teaching point: if this agent writes an essay, the whole pipeline degrades.
issue_classifier = Agent(
    role="Issue Classification Specialist",
    goal="Classify the customer's message into a structured JSON with 4 fields: type, urgency, sentiment, summary.",
    backstory=(
        "You are a customer intent analyst. You NEVER write prose. "
        "You ALWAYS output valid JSON with exactly these keys: "
        "type (billing/technical/login/refund/general), "
        "urgency (low/normal/high/critical), "
        "sentiment (satisfied/neutral/confused/angry), "
        "summary (one sentence, under 15 words). "
        "Nothing else. Just the JSON."
    ),
    llm=llm,
    verbose=False
)

# Agent 2: Knowledge agent — reads type/urgency from Agent 1's JSON
issue_classifier_note = '''
# NOTE FOR INSTRUCTOR:
# knowledge_agent uses the OUTPUT of issue_classifier as context.
# It reads 'type' (e.g. 'billing') to narrow the knowledge base search.
# This is task chaining in action.
'''

knowledge_agent = Agent(
    role="Knowledge Base Specialist",
    goal="Find the best-matching solution for the classified issue type from the support knowledge base.",
    backstory=(
        "You search internal support documentation given a classified issue type. "
        "You return: solution title + 3 numbered resolution steps. "
        "If no relevant solution exists, you return exactly: NO MATCH FOUND. "
        "You do not invent solutions that are not in the knowledge base."
    ),
    llm=llm,
    verbose=False
)

# Agent 3: Solution writer — formats the response for the customer
solution_agent = Agent(
    role="Customer Response Writer",
    goal="Write a clear, empathetic, numbered customer support response using the KB solution.",
    backstory=(
        "You write customer-facing support replies. "
        "Every response: opens with an empathetic acknowledgement, "
        "provides numbered steps, closes with an offer to help further. "
        "Plain language only — no technical jargon. Maximum 200 words."
    ),
    llm=llm,
    verbose=False
)

# Agent 4: Escalation — governance checkpoint
# Teaching point: this is the human-in-the-loop mechanism at the agent layer.
# It decides BEFORE the response is sent whether a human must review.
escalation_agent = Agent(
    role="Escalation Coordinator",
    goal="Determine whether the AI response is sufficient or requires human agent review.",
    backstory=(
        "You are the quality gate. Before any AI response is sent to a customer, "
        "you check: confidence level, urgency, and whether the KB returned a match. "
        "If urgency='critical' OR knowledge='NO MATCH FOUND' OR sentiment='angry' AND type='billing': "
        "you escalate. Otherwise you approve. You output exactly: RESOLVED or ESCALATED."
    ),
    llm=llm,
    verbose=False
)


In [ ]:
# ── Task definitions — notice how expected_output shapes the pipeline ──

# TASK 1: Output is a JSON string — this is what knowledge_agent reads
identify_issue = Task(
    description=(
        'Classify this customer message: "{customer_query}"\n\n'
        "Output ONLY valid JSON with exactly these keys:\n"
        "  type     : billing | technical | login | refund | general\n"
        "  urgency  : low | normal | high | critical\n"
        "  sentiment: satisfied | neutral | confused | angry\n"
        "  summary  : one sentence under 15 words describing the core issue\n\n"
        "Example: {\"type\": \"billing\", \"urgency\": \"high\", "
        "\"sentiment\": \"angry\", \"summary\": \"Customer charged twice for subscription\"}"
    ),
    # This JSON format is the CONTRACT — knowledge_agent depends on it
    expected_output='JSON object: {"type": str, "urgency": str, "sentiment": str, "summary": str}',
    agent=issue_classifier
)

# TASK 2: Uses type from TASK 1's JSON to search the knowledge base
fetch_knowledge = Task(
    description=(
        'Given the classified issue and original message: "{customer_query}"\n'
        "Search the knowledge base for the best-matching resolution.\n\n"
        "If match found: return solution title + 3 numbered steps.\n"
        "If no match:    return exactly 'NO MATCH FOUND' — nothing else."
    ),
    expected_output="Solution title + 3 numbered resolution steps, OR exactly 'NO MATCH FOUND'",
    agent=knowledge_agent
)

# TASK 3: Writes customer-facing text using all context so far
generate_response = Task(
    description=(
        'Write a customer support response for: "{customer_query}"\n'
        "Using the classification and KB solution above:\n\n"
        "Structure:\n"
        "  Line 1   : Empathetic acknowledgement (reference their specific issue)\n"
        "  Lines 2-5: Numbered resolution steps\n"
        "  Last line: 'If this doesn\'t resolve your issue, reply here and I\'ll escalate to a specialist.'\n\n"
        "Maximum 200 words. Plain language — no jargon."
    ),
    expected_output="Customer-facing support response: acknowledgement + numbered steps + follow-up offer",
    agent=solution_agent
)

# TASK 4: Decision gate — approve or escalate before message is sent
# Teaching point: the escalation agent reads EVERY previous output
# before deciding. This is the human-in-the-loop proxy.
apply_escalation = Task(
    description=(
        "Review the complete support interaction for: {customer_query}\n\n"
        "Escalate if ANY of these are true:\n"
        "  • urgency = 'critical'\n"
        "  • Knowledge base returned 'NO MATCH FOUND'\n"
        "  • sentiment = 'angry' AND type = 'billing' or 'refund'\n\n"
        "Output format (pick ONE):\n"
        "  RESOLVED: [one-line confirmation — response approved to send]\n"
        "  ESCALATED: [reason for escalation] | [notes for human agent reviewing this case]"
    ),
    expected_output="Either 'RESOLVED: [confirmation]' or 'ESCALATED: [reason] | [notes for human]'",
    agent=escalation_agent
)

# ── Build the Crew ────────────────────────────────────────────
support_crew = Crew(
    agents=[issue_classifier, knowledge_agent, solution_agent, escalation_agent],
    tasks=[identify_issue, fetch_knowledge, generate_response, apply_escalation],
    verbose=False
)

# ── 4 test cases — try each one ──────────────────────────────
test_cases = {
    "login":   "I'm locked out of my account even though I reset my password twice. This is urgent.",
    "billing": "You charged me $199 twice this month. I never authorised a second charge. Refund it NOW.",
    "bug":     "The app crashes every time I tap the dashboard icon after your latest update.",
    "simple":  "Do you offer annual plans? Is there a discount for paying yearly?",
}

active = test_cases["billing"]   # Change the key to test other cases

print(f"Customer message: '{active}'\n")
print("Running 4-agent support pipeline...\n")

result = support_crew.kickoff(inputs={"customer_query": active})

print("\n" + "═"*62)
print("  FINAL PIPELINE DECISION")
print("═"*62)
print(result)


In [ ]:
# ── EXPERIMENT: Remove expected_output and see what breaks ────
# This is the most important 5-minute experiment of Block 4.

print("EXPERIMENT: What happens without expected_output?")
print("=" * 55)
print()

broken_classifier = Agent(
    role="Issue Classifier (broken version)",
    goal="Classify the customer issue.",
    backstory="You analyse customer messages and describe the issue type.",
    llm=llm, verbose=False
)

# NO expected_output field — watch what the agent produces
broken_task = Task(
    description='Classify this customer message: "{customer_query}"',
    # expected_output is MISSING — the agent writes whatever it wants
    expected_output="Description of the customer issue",   # vague → essay output
    agent=broken_classifier
)

broken_crew = Crew(
    agents=[broken_classifier],
    tasks=[broken_task],
    verbose=False
)

result_broken = broken_crew.kickoff(inputs={"customer_query": test_cases["billing"]})

print("Broken classifier output (no JSON contract):")
print(result_broken)
print()
print("Now try to extract 'urgency' programmatically from that text.")
print("You can't — reliably. That is why expected_output is not optional.")

# Try to extract urgency
import re
urgency_match = re.search(r'"urgency"\s*:\s*"(\w+)"', str(result_broken))
print(f"\nParsed urgency: {urgency_match.group(1) if urgency_match else 'NOT FOUND — pipeline broken'}")


---
### ★ Exercise — Add the 5th Agent

Add a `satisfaction_predictor` between `solution_agent` and `escalation_agent`:
- Role: "Customer Satisfaction Predictor"
- Goal: Predict satisfaction score 1–10 before the response is sent
- `expected_output`: `{"predicted_score": int, "key_risk": str}`

This is the **Level 1 practice assignment**.

---


---
# Part 4 — Healthcare Triage: All Four Planning Modes in One System
### PPT: Block 5 — Slides 26–28 | ⏱ 20 minutes | Requires API key

**This is the most important live demo of Day 1.**
Every planning concept we have covered maps to one agent:

| Agent | Planning Mode | Real-world equivalent |
|-------|-------------|----------------------|
| `triage_agent` | **Reactive** | ER nurse — reads patient fresh, no pre-built script |
| `planning_agent` | **Hierarchical** | Attending physician — decomposes "treat" into steps |
| `decision_agent` | **Adaptive Goal** | ICU coordinator — changes plan when ICU is full |
| `monitoring_agent` | **Reinforcement** | Patient safety officer — watches, flags, closes loop |


In [ ]:
# ═══════════════════════════════════════════════════════════
#  PART 4 — Healthcare Triage: All Four Planning Modes
#  Each agent is annotated with its planning type.
# ═══════════════════════════════════════════════════════════

from crewai import Agent, Task, Crew

# ── REACTIVE PLANNING: triage_agent ──────────────────────────
# Reactive = no pre-built plan. Reads current patient context fresh.
# Like a triage nurse: doesn't decide the plan before seeing the patient.
# Every patient is assessed independently — no history bias.
triage_agent = Agent(
    role="Emergency Triage Specialist",
    goal="Classify patient urgency based only on the presented symptoms. Output: severity + immediate action.",
    backstory=(
        "You follow ESI (Emergency Severity Index) triage protocols. "
        "You NEVER use patient history to bias classification — you assess what is in front of you now. "
        "Severity levels: Low | Moderate | High | Emergency. "
        "You recommend ONE immediate action and one monitoring trigger."
    ),
    llm=llm,
    verbose=False
)

# ── HIERARCHICAL PLANNING: planning_agent ────────────────────
# Hierarchical = decomposes 'treatment' into ordered sub-steps.
# Like an attending physician: builds a structured care plan.
planning_agent = Agent(
    role="Care Planning Physician",
    goal="Generate a structured, time-bounded treatment plan with numbered steps and monitoring intervals.",
    backstory=(
        "You build evidence-based care plans. Every plan: "
        "immediate actions (0-15 min) → stabilisation (15-60 min) → monitoring plan → escalation triggers. "
        "Steps are numbered and time-bounded. You do not improvise — you follow clinical frameworks."
    ),
    llm=llm,
    verbose=False
)

# ── ADAPTIVE GOAL PLANNING: decision_agent ───────────────────
# Adaptive = goal changes based on real-time resource state.
# Like an ICU coordinator: when ICU is full, 'Moderate' becomes 'Urgent'.
# The OBJECTIVE of care changes — not just the method.
decision_agent = Agent(
    role="Resource-Aware Clinical Decision Unit",
    goal=(
        "Select treatment mode based on triage severity AND current resource availability. "
        "If ICU occupancy > 80%: upgrade Moderate cases to Urgent. Justify every decision."
    ),
    backstory=(
        "You have access to real-time hospital resource state. "
        "You balance clinical need with resource constraints. "
        "You ALWAYS document your resource state assumption and its impact on your decision. "
        "A physician will review this — you must justify every decision in 2 sentences."
    ),
    llm=llm,
    verbose=False
)

# ── REINFORCEMENT / MONITORING: monitoring_agent ─────────────
# Reinforcement = watches other agents, closes the feedback loop.
# This agent does NOT treat the patient. It watches and validates.
# Teaching point: this is the SAFETY LAYER — the human-in-the-loop proxy.
# If this agent flags something, a human physician reviews BEFORE any action.
monitoring_agent = Agent(
    role="Patient Safety & Governance Monitor",
    goal=(
        "Validate ALL previous agent outputs against the patient record. "
        "Flag any recommendation that: contradicts listed allergies, "
        "conflicts with triage severity, or recommends an action without a monitoring plan."
    ),
    backstory=(
        "You are the final safety check before any recommendation reaches a physician. "
        "You have zero tolerance for allergy contradictions. "
        "Output EXACTLY one of: "
        "SAFETY FLAG: [specific issue found] "
        "— OR — "
        "CLEARED: all outputs consistent. Recommendation approved for physician review."
    ),
    llm=llm,
    verbose=False
)


In [ ]:
# ── Task definitions — each maps to a specific planning mode ─

# REACTIVE: triage task — reads patient fresh, no history bias
triage_task = Task(
    description=(
        "Evaluate this patient presentation:\n\n"
        "{patient_details}\n\n"
        "Classify using ESI protocol:\n"
        "  1. List the 3-5 most significant clinical indicators\n"
        "  2. Classify severity: Low | Moderate | High | Emergency\n"
        "  3. State ONE immediate action (e.g. 'Start IV access and cardiac monitoring')\n"
        "  4. State ONE monitoring trigger (e.g. 'Escalate if SaO2 drops below 94%')\n\n"
        "Output as a structured triage report — no paragraphs."
    ),
    expected_output="Triage report: clinical indicators, severity classification, immediate action, monitoring trigger",
    agent=triage_agent
)

# HIERARCHICAL: care plan task — decomposes treatment into steps
care_plan_task = Task(
    description=(
        "Using the triage result, generate a structured care plan for: {patient_details}\n\n"
        "Produce:\n"
        "  Immediate actions (0-15 min): numbered list\n"
        "  Stabilisation plan (15-60 min): numbered list\n"
        "  Monitoring plan: what to track, frequency, normal ranges\n"
        "  Escalation trigger: one sentence — what deterioration looks like\n\n"
        "All steps must be time-bounded. No open-ended instructions."
    ),
    expected_output="Structured care plan: immediate actions + stabilisation + monitoring + escalation trigger",
    agent=planning_agent
)

# ADAPTIVE GOAL: decision task — objective changes based on resource state
decision_task = Task(
    description=(
        "Using triage result and care plan for: {patient_details}\n\n"
        "SIMULATED HOSPITAL RESOURCE STATE:\n"
        "  ICU occupancy  : 85% (above threshold)\n"
        "  Available beds : 2 standard, 0 ICU\n"
        "  On-call staff  : 1 senior physician, 2 nurses\n\n"
        "Decision rules (apply in order):\n"
        "  1. If severity=Emergency           → REACTIVE EMERGENCY: immediate senior physician\n"
        "  2. If severity=High AND ICU > 80%  → URGENT CARE: treat in standard bed, escalate ICU waitlist\n"
        "  3. If severity=Moderate            → DELIBERATE PLANNED CARE: standard admission\n"
        "  4. If severity=Low                 → AMBULATORY CARE: treat and discharge with follow-up\n\n"
        "Output: Decision mode + 2-sentence justification including resource constraint impact."
    ),
    expected_output="Treatment mode decision + 2-sentence justification including resource state reference",
    agent=decision_agent
)

# REINFORCEMENT/MONITORING: validation — closes the feedback loop
monitoring_task = Task(
    description=(
        "Validate ALL previous outputs for patient: {patient_details}\n\n"
        "CHECK LIST (validate every item):\n"
        "  □ Triage severity matches severity of symptoms described\n"
        "  □ Care plan does not recommend any drug that appears in patient allergy history\n"
        "  □ Decision mode is consistent with triage severity and resource state\n"
        "  □ An escalation trigger is defined in the care plan\n"
        "  □ At least one monitoring interval is specified\n\n"
        "AUTOMATIC ESCALATION TRIGGERS (output SAFETY FLAG if found):\n"
        "  • Any recommendation for drug X when patient listed X as allergy\n"
        "  • Care plan classifies patient as Low when symptoms include: chest pain / loss of consciousness / breathing difficulty\n\n"
        "Output EXACTLY one of:\n"
        "  SAFETY FLAG: [specific item from checklist that failed] — [recommended action]\n"
        "  CLEARED: all 5 checklist items passed. Recommendation approved for physician review."
    ),
    expected_output="SAFETY FLAG: [detail] OR CLEARED: [confirmation of all 5 checklist items]",
    agent=monitoring_agent
)


In [ ]:
# ── Build the healthcare triage crew ─────────────────────────

triage_crew = Crew(
    agents=[triage_agent, planning_agent, decision_agent, monitoring_agent],
    tasks=[triage_task, care_plan_task, decision_task, monitoring_task],
    verbose=False
)

# ── Patient cases — each tests a different interaction type ───
patients = {
    # Methotrexate + Ibuprofen — pediatric, low WBC
    "pediatric": (
        "Alex Johnson, 8 years old. Diagnosis: Juvenile Idiopathic Arthritis. "
        "Medications: Methotrexate 10mg/week, Ibuprofen 200mg PRN. "
        "Presenting: low-grade fever 38.1°C, joint swelling both knees, fatigue. "
        "Lab result: WBC 2.8 (LOW). Documented allergy: NONE on file."
    ),
    # Naproxen in 3rd trimester — pregnancy risk
    "pregnancy": (
        "Sarah Martinez, 31 years old, 28 weeks pregnant (first pregnancy). "
        "Self-prescribed Naproxen 250mg for back pain. "
        "Presenting: back pain, BP 145/92 (elevated), mild headache. "
        "Documented allergy: Codeine (nausea). No prior pregnancy complications."
    ),
    # CKD + Metformin + Ibuprofen — renal clearance risk
    "renal": (
        "Robert Chen, 58 years old. Diagnoses: CKD Stage 3, Type 2 Diabetes, Hypertension. "
        "Medications: Metformin 1000mg BD, Lisinopril 10mg OD, Ibuprofen 400mg TDS (new). "
        "Labs: eGFR 42 mL/min (reduced). Presenting: fatigue, nausea, bilateral leg oedema. "
        "Documented allergy: Sulfonamides (rash)."
    ),
    # Warfarin + Aspirin — additive bleed risk + history
    "elderly": (
        "Margaret Thompson, 78 years old. AF diagnosis, prior GI bleed December 2023. "
        "Medications: Warfarin 5mg daily (INR 2.4), Aspirin 75mg (added last week by GP). "
        "Presenting: mild epigastric discomfort, no haematemesis, BP 128/76. "
        "Documented allergy: Penicillin (anaphylaxis)."
    ),
    # Clarithromycin + Clopidogrel — CYP3A4 inhibition cascade
    "cardiac": (
        "David Okafor, 52 years old. Post-MI 6 weeks. "
        "Medications: Clopidogrel 75mg, Atorvastatin 40mg, Aspirin 75mg. "
        "New GP prescription: Clarithromycin 500mg BD for community pneumonia. "
        "Presenting: mild chest tightness, productive cough, temp 38.3°C. "
        "Documented allergy: None."
    ),
}

# ── Select the patient to run ─────────────────────────────────
# Change this key to test different cases:
# 'pediatric' | 'pregnancy' | 'renal' | 'elderly' | 'cardiac'
active_patient = "cardiac"

print(f"Processing patient case: {active_patient.upper()}")
print(f"Details: {patients[active_patient][:100]}...\n")
print("This is the most complex interaction to detect (CYP3A4 inhibition cascade).")
print("Watch whether the monitoring_agent catches the Clarithromycin + Clopidogrel conflict.\n")

result = triage_crew.kickoff(inputs={"patient_details": patients[active_patient]})

print("\n" + "═"*62)
print("  COMPLETE TRIAGE PIPELINE OUTPUT")
print("═"*62)
print(result)


---
### ★ Group Lab Exercise (25 min)

For the patient you just ran, answer these questions:

1. **Agents**: What 2 additional agents would make this pipeline safer?
2. **Goal parameter**: Write the exact CrewAI `goal=` string for each new agent
3. **Failure mode**: What is the one thing `monitoring_agent` could miss with your patient?
4. **Human checkpoint**: At which exact task must a physician approve before execution?

**Run the 'renal' case**: `active_patient = "renal"`
Does the monitoring agent flag the Metformin + Ibuprofen + CKD interaction?

---


---
# Part 5 — Reinforcement Loop (Framework-Free Python)
### PPT: Block 7 — Slides 32–40 | ⏱ 20 minutes | Requires API key

**Why no framework?**
CrewAI and LangChain abstract the reinforcement loop.
This section shows the raw mechanics — so you understand what the frameworks do internally.
Once you see it here, you'll recognise it inside every agent framework.

**The loop:**
```
ACT → OBSERVE → ADAPT → (better) ACT → OBSERVE → ADAPT → ...
```

**Three components of any reinforcement agent:**
- `strategy` — the current behaviour policy
- `learning_history` — memory of every outcome
- `success_rate` — rolling performance score


In [ ]:
# ═══════════════════════════════════════════════════════════
#  PART 5 — Reinforcement Loop (pure Python + LLM)
#  No CrewAI. Shows the raw mechanics of the feedback loop.
# ═══════════════════════════════════════════════════════════

from langchain.schema import HumanMessage, SystemMessage

class ReinforcementAgent:
    '''
    Minimal reinforcement loop agent.

    The three core attributes mirror every RL framework:
      strategy        = policy (what the agent currently believes works)
      learning_history = replay buffer (record of every action-outcome pair)
      success_rate    = value function (rolling estimate of performance)

    The three-step loop:
      1. ACT    — generate response using current strategy as system prompt
      2. OBSERVE — receive feedback score (1-5) from environment
      3. ADAPT  — if success_rate < threshold, rewrite strategy using LLM
    '''

    def __init__(self, name: str, initial_strategy: str):
        self.name = name
        self.strategy = initial_strategy
        self.learning_history = []
        self.success_rate = 0.5     # Start neutral — no prior data

    # ── STEP 1: ACT ──────────────────────────────────────────
    # The strategy IS the system prompt.
    # Changing the strategy changes every future action.
    def act(self, situation: str) -> str:
        '''Generate a response using current strategy as context.'''
        response = llm.invoke([
            # Strategy = agent's current 'policy' injected as system context
            # This is equivalent to updating agent backstory in CrewAI
            SystemMessage(content=f"Your current operating strategy: {self.strategy}"),
            HumanMessage(content=f"Handle this situation: {situation}"),
        ])
        return response.content

    # ── STEP 2: OBSERVE ──────────────────────────────────────
    # Record WHAT happened and HOW WELL it went.
    # In production: feedback_score comes from user ratings, outcomes, downstream metrics.
    # In this demo: we simulate it manually.
    def observe(self, feedback_score: int, details: str) -> dict:
        '''Record outcome. feedback_score: 1 (failure) → 5 (excellent).'''
        entry = {
            "situation":     None,   # filled by caller if needed
            "feedback_score": feedback_score,
            "details":        details,
            "strategy_used":  self.strategy,  # snapshot — important for debugging
        }
        self.learning_history.append(entry)
        return entry

    # ── STEP 3: ADAPT ─────────────────────────────────────────
    # If success_rate is low: ask LLM to rewrite the strategy.
    # The strategy rewrite IS the learning step.
    # In CrewAI: equivalent to updating agent.backstory between crew runs.
    def adapt(self) -> str:
        '''Update success_rate. If low, rewrite strategy using failure analysis.'''
        recent = self.learning_history[-5:]   # rolling window of last 5
        if not recent:
            return "No data yet."

        self.success_rate = sum(1 for e in recent if e["feedback_score"] >= 4) / len(recent)

        if self.success_rate < 0.6:
            # Extract recent failures to analyse
            failures = [
                e for e in self.learning_history[-3:]
                if e["feedback_score"] <= 2
            ]
            failure_summary = (
                "\n".join(f"  - {f['details']}" for f in failures)
                if failures else "  - Recent scores were low overall."
            )

            # Ask LLM to improve strategy — this IS the learning step
            new_strategy_response = llm.invoke([
                SystemMessage(content=(
                    "You improve AI agent strategies based on failure patterns. "
                    "Output ONLY the improved strategy in 2-3 sentences. "
                    "No preamble. No headers. Just the strategy."
                )),
                HumanMessage(content=(
                    f"Current strategy: {self.strategy}\n\n"
                    f"Recent failures:\n{failure_summary}\n\n"
                    "Write an improved strategy that addresses these failures:"
                )),
            ])

            old_strategy = self.strategy[:60]
            self.strategy = new_strategy_response.content.strip()

            return (
                f"⟳ STRATEGY UPDATED  (success rate was {self.success_rate:.0%})\n"
                f"  WAS: {old_strategy}...\n"
                f"  NOW: {self.strategy[:80]}..."
            )
        else:
            return f"✓ Strategy maintained  (success rate {self.success_rate:.0%} — above threshold)"

    def print_history(self):
        '''Print a readable summary of all learning history.'''
        print(f"\n{'─'*58}")
        print(f"  {self.name} | Total runs: {len(self.learning_history)} | Success rate: {self.success_rate:.0%}")
        print(f"{'─'*58}")
        for i, e in enumerate(self.learning_history, 1):
            icon = "✓" if e["feedback_score"] >= 4 else "✗"
            print(f"  Run {i:02d}: {icon}  Score {e['feedback_score']}/5  —  {e['details'][:55]}")
        print(f"\n  Final strategy: {self.strategy[:100]}...")
        print(f"{'─'*58}")

print("ReinforcementAgent class defined. ✓")
print("Three attributes: strategy | learning_history | success_rate")
print("Three methods:    act()    | observe()         | adapt()")


In [ ]:
# ── Demo: 6-cycle reinforcement loop ─────────────────────────
# Each cycle: ACT → OBSERVE → ADAPT
# Watch how the strategy changes based on feedback.

agent = ReinforcementAgent(
    name="CustomerSupportBot-v1",
    initial_strategy="Provide factual, professional information about company policies."
)

scenarios = [
    # (situation, feedback_score, feedback_reason)
    (
        "Customer: My order #99123 is 5 days late. I needed it for my daughter's birthday tomorrow!",
        1,
        "Too formal and policy-focused — ignored the emotional distress completely"
    ),
    (
        "Customer: I was charged $199 twice this month. I never authorised a second charge!",
        2,
        "Asked customer to 'verify with their bank' — shifted responsibility instead of owning the error"
    ),
    (
        "Customer: Can you explain the difference between your Standard and Premium plan?",
        5,
        "Clear, concise feature comparison — customer replied 'thank you, exactly what I needed'"
    ),
    (
        "Customer: My account was hacked and someone made purchases I didn't authorise!",
        4,
        "Empathetic, offered to lock account immediately — customer felt heard and safe"
    ),
    (
        "Customer: I've been a customer for 10 years and this is the worst service I've ever received.",
        1,
        "Response defended the company instead of acknowledging the long-term relationship"
    ),
    (
        "Customer: I can't find where to download my invoice for accounting purposes.",
        5,
        "Gave precise steps to navigate to billing section — solved in one reply"
    ),
]

print(f"{'═'*58}")
print(f"  Reinforcement Loop Demo  —  6 Cycles")
print(f"{'═'*58}")

for i, (situation, score, reason) in enumerate(scenarios, 1):
    print(f"\n{'─'*58}")
    print(f"  CYCLE {i} of 6")
    print(f"{'─'*58}")
    print(f"  Situation  : {situation[:65]}...")
    print(f"  Strategy   : {agent.strategy[:65]}...")

    # STEP 1: ACT
    response = agent.act(situation)
    print(f"  Response   : {response[:90]}...")

    # STEP 2: OBSERVE — in production this comes from the environment
    agent.observe(score, reason)
    icon = "✓" if score >= 4 else "✗"
    print(f"  Feedback   : {icon}  {score}/5  —  {reason}")

    # STEP 3: ADAPT — strategy may or may not change
    adaptation = agent.adapt()
    print(f"  Adapt      : {adaptation[:100]}")

# Final summary
agent.print_history()


In [ ]:
# ── CRITICAL INSIGHT: What if the reward signal is wrong? ────
#
# This is the most important 5 minutes of Block 7.
# We change the reward signal to SPEED (1 score regardless of quality)
# and watch the agent 'learn' the wrong thing perfectly.

print("EXPERIMENT: Wrong Reward Signal")
print("=" * 55)
print("Scenario: we reward speed — score=5 for any quick response.")
print("Watch what the agent 'learns'.\n")

speed_agent = ReinforcementAgent(
    name="SpeedBot (wrong reward)",
    initial_strategy="Provide helpful, empathetic customer support."
)

# All feedback is based ONLY on response speed (ignoring quality)
speed_scenarios = [
    ("Customer: My order is 3 weeks late and I'm furious!",
     5, "SPEED: responded in under 2 seconds"),   # ← wrong metric
    ("Customer: I was overcharged — please explain!",
     5, "SPEED: quick reply"),
    ("Customer: The app is completely broken!",
     5, "SPEED: immediate response"),
]

for i, (sit, score, reason) in enumerate(speed_scenarios, 1):
    response = speed_agent.act(sit)
    speed_agent.observe(score, reason)
    adaptation = speed_agent.adapt()
    print(f"Cycle {i}: Feedback={score}/5 ({reason})")
    print(f"  Response: {response[:80]}...")
    print(f"  Adapt: {adaptation[:80]}\n")

print("Final strategy:")
print(speed_agent.strategy)
print()
print("The agent is now highly optimised for SPEED.")
print("It will give fast responses regardless of quality.")
print("This is a perfectly optimised agent — optimised for the wrong thing.")
print()
print("In a clinical triage system: 'respond fast' → classify everyone as Low → miss emergencies.")
print("The reward function IS the governance decision.")


---
### ★ Exercise — Extend the Loop

Modify `adapt()` to:
1. Store the **previous strategy** in `learning_history` alongside the new one
2. Add a `rollback()` method that restores the strategy with the highest success_rate ever recorded
3. Run 10 cycles and show the strategy change history as a timeline

This is the **Level 3 practice assignment** skeleton.

---


---
# Part 6 — Hybrid Agents: The 3-Layer Architecture
### PPT: Block 8 — Slides 41–44 | ⏱ 20 minutes | Requires API key

**ReAct = Reason + Act**

```
Standard agent: Think → Act → Done
ReAct agent:    Think → Act → Observe → Think → Act → Observe → ...
```

Every tool call + result-read is **one ReAct loop**.
The agent doesn't commit to a plan and execute blindly.
It reasons about each intermediate result before the next step.

**3-Layer Model:**
```
Layer 1 (Strategic)    → Crew definition — what happens, in what order
Layer 2 (Reactive)     → Agents — how they do it, adapting to context
Layer 3 (Reinforcement)→ Governance agent — watches, scores, closes loop
```


In [ ]:
# ═══════════════════════════════════════════════════════════
#  PART 6 — Hybrid ReAct: 3-Layer Clinical Architecture
# ═══════════════════════════════════════════════════════════

from crewai import Agent, Task, Crew

# ── LAYER 2 AGENTS: Reactive execution specialists ───────────
# Each reads its task context and adapts. No pre-built answers.
# They implement the ReAct loop internally — every tool call
# is a Think → Act → Observe cycle within the agent.

symptom_agent = Agent(
    role="Clinical Symptom Extraction Specialist",
    goal=(
        "Extract and structure every clinically relevant item from the patient report. "
        "Produce a structured list: chief complaint, symptoms, vitals, red flags, current medications, allergies."
    ),
    backstory=(
        "You are a clinical documentation specialist. You extract data — you do not diagnose. "
        "You never infer. If a vital sign is not stated: report 'not documented'. "
        "Your output is the input to a drug interaction analyst — format matters."
    ),
    llm=llm,
    verbose=False
)

risk_agent = Agent(
    role="Clinical Pharmacist — Drug Interaction Analyst",
    goal=(
        "Identify every drug-drug interaction and drug-condition contraindication in the patient's record. "
        "Rate severity: MINOR | MODERATE | SEVERE | CONTRAINDICATED."
    ),
    backstory=(
        "You are a senior clinical pharmacist with specialisation in drug interaction databases. "
        "You check EVERY medication pair. You flag EVERY interaction regardless of severity. "
        "Format: for each interaction: Drug A + Drug B → Mechanism → Clinical impact → Severity rating. "
        "If no interactions found: state 'No clinically significant interactions identified' and explain why."
    ),
    llm=llm,
    verbose=False
)

# ── LAYER 1 AGENT: Strategic integrator ──────────────────────
# The orchestrator represents the Crew's strategic intent.
# It reads all Layer 2 outputs and produces the integrated recommendation.
orchestrator = Agent(
    role="Attending Physician — Clinical Recommendation Integrator",
    goal=(
        "Using symptom extraction and drug interaction analysis, "
        "produce a structured clinical recommendation ready for physician review."
    ),
    backstory=(
        "You are a senior attending physician AI. You read the work of your specialist colleagues "
        "and integrate it into one actionable recommendation. "
        "You ALWAYS include: primary concern, immediate actions, medication adjustments with rationale, "
        "monitoring plan, and escalation trigger. "
        "You explicitly note where physician discretion is required."
    ),
    llm=llm,
    verbose=False
)

# ── LAYER 3 AGENT: Governance and feedback ───────────────────
# This agent closes the reinforcement loop.
# Its quality_score is the reward signal for the next iteration.
# Teaching point: this agent's job is to make the system self-improving.
governance_agent = Agent(
    role="Clinical Governance & Safety Validator",
    goal=(
        "Validate all clinical outputs for safety, consistency, and guideline compliance. "
        "Output a quality_score (1-10) and a governance verdict: APPROVED | FLAGGED | ESCALATE."
    ),
    backstory=(
        "You are the clinical governance AI. You have zero tolerance for: "
        "allergy contradictions, missing escalation triggers, and severity mismatches. "
        "For every output you review, you assign a quality_score: "
        "  10 = perfect, all items consistent, no risks "
        "  7-9 = good, minor gaps noted "
        "  4-6 = acceptable, physician attention needed "
        "  1-3 = safety issue, escalate immediately "
        "Your output MUST include: STATUS (APPROVED/FLAGGED/ESCALATE) + quality_score: [n] + 2-sentence reasoning."
    ),
    llm=llm,
    verbose=False
)


In [ ]:
# ── Task definitions: Layer 2 feeds Layer 3 ──────────────────

extract_task = Task(
    description=(
        "Extract all clinically relevant data from:\n\n"
        "{patient_report}\n\n"
        "Output structure (use these exact headings):\n"
        "  Chief complaint   : [one sentence]\n"
        "  Symptoms          : [bulleted list with duration and severity]\n"
        "  Vital signs       : [list — if not documented say 'not documented']\n"
        "  Current medications: [name, dose, frequency, duration — one per line]\n"
        "  Known allergies   : [list — if none say 'None documented']\n"
        "  Red flags present : [list any immediate clinical concerns]"
    ),
    expected_output="Structured extraction using the 6 exact headings above",
    agent=symptom_agent
)

interaction_task = Task(
    description=(
        "Using the structured extraction for: {patient_report}\n\n"
        "For each medication pair (check ALL combinations):\n"
        "  Drug A + Drug B → Mechanism of interaction → Clinical impact → Severity\n\n"
        "Also check each medication against the patient's diagnoses for contraindications.\n\n"
        "If any interaction rated SEVERE or CONTRAINDICATED: flag it first with ⚠ PRIORITY FLAG.\n"
        "If no interactions: state 'No clinically significant interactions identified' + explanation."
    ),
    expected_output="Interaction table: drug pairs + mechanism + impact + severity. Priority flags first.",
    agent=risk_agent
)

recommendation_task = Task(
    description=(
        "Using symptom extraction and interaction analysis for: {patient_report}\n\n"
        "Produce a clinical recommendation with these sections:\n"
        "  Primary clinical concern  : [one sentence — the most important issue]\n"
        "  Immediate actions (0-30min): [numbered list, time-bounded]\n"
        "  Medication adjustments    : [drug changes with clinical rationale]\n"
        "  Monitoring plan           : [what to monitor, frequency, alert thresholds]\n"
        "  Escalation trigger        : [one sentence — when to call the senior physician]\n"
        "  Physician discretion note : [what requires human clinical judgement]"
    ),
    expected_output="Structured clinical recommendation with all 6 sections completed",
    agent=orchestrator
)

# Layer 3: Governance validates ALL previous outputs
governance_task = Task(
    description=(
        "Review the complete clinical pathway for: {patient_report}\n\n"
        "Validate checklist (check each item explicitly):\n"
        "  □ Symptom extraction is complete — no obvious omissions\n"
        "  □ ALL SEVERE/CONTRAINDICATED interactions are addressed in the recommendation\n"
        "  □ No recommended medication appears in patient's allergy list\n"
        "  □ An escalation trigger is defined\n"
        "  □ Physician discretion is noted for subjective decisions\n\n"
        "Output format — use EXACTLY this structure:\n"
        "  STATUS: APPROVED | FLAGGED | ESCALATE\n"
        "  quality_score: [1-10]\n"
        "  reasoning: [2 sentences]\n"
        "  human_review_required: yes | no — [reason if yes]"
    ),
    expected_output="STATUS + quality_score + reasoning + human_review decision",
    agent=governance_agent
)


In [ ]:
# ── Build and run the 3-layer hybrid crew ────────────────────

hybrid_crew = Crew(
    agents=[symptom_agent, risk_agent, orchestrator, governance_agent],
    tasks=[extract_task, interaction_task, recommendation_task, governance_task],
    verbose=False
)

# Most complex case: CYP3A4 inhibition cascade
cardiac_patient = (
    "David Okafor, 52 years old. Post-MI 6 weeks. "
    "Medications: Clopidogrel 75mg daily, Atorvastatin 40mg daily, Aspirin 75mg daily. "
    "New GP prescription today: Clarithromycin 500mg twice daily for community-acquired pneumonia. "
    "Presenting: mild chest tightness, productive cough, temperature 38.3°C. "
    "Documented allergies: None. eGFR 72 (normal). Last ECG: sinus rhythm, no acute changes."
)

print("3-layer hybrid clinical crew running...")
print(f"Patient: {cardiac_patient[:90]}\n")
print("Layer 2 agents run first (symptom extraction + interaction analysis).")
print("Layer 1 orchestrator integrates their output.")
print("Layer 3 governance validates and scores the full pathway.\n")

result = hybrid_crew.kickoff(inputs={"patient_report": cardiac_patient})

print("\n" + "═"*62)
print("  FULL 3-LAYER CLINICAL PATHWAY OUTPUT")
print("═"*62)
print(result)


---
# Part 7 — Capstone: Connecting the Reinforcement Loop to the Crew
### PPT: Assignment Level 3 | ⏱ 25 min group exercise | Requires API key

**The complete picture:**
- The Crew runs the pipeline (3-layer architecture)
- The governance_agent scores the output (reward signal)
- The score feeds back into the risk_agent's strategy (policy update)
- Next run starts with an improved risk_agent

**This is the system that improves itself.**


In [ ]:
# ═══════════════════════════════════════════════════════════
#  CAPSTONE — Reinforcement Loop Connected to the Crew
#  Level 3 Practice Assignment skeleton.
# ═══════════════════════════════════════════════════════════

import re

class ClinicalPipelineWithRL:
    '''
    Wraps the 3-layer hybrid crew with a reinforcement feedback loop.

    Each iteration:
      1. Run the crew with current risk_agent_strategy
      2. Extract quality_score from governance_agent output
      3. If score < 7: ask LLM to improve risk_agent_strategy
      4. Next iteration uses the improved strategy

    This connects Part 5 (raw RL loop) to Part 6 (hybrid crew).
    The governance_agent IS the observer. The risk_agent strategy IS the policy.
    '''

    def __init__(self):
        # Starting strategy for risk_agent — will evolve with each iteration
        self.risk_strategy = (
            "You are a senior clinical pharmacist. Check every medication pair for interactions. "
            "Rate severity: MINOR | MODERATE | SEVERE | CONTRAINDICATED. "
            "Flag SEVERE and CONTRAINDICATED interactions first."
        )
        # Learning history: one entry per iteration
        self.learning_history = []

    def _build_crew(self) -> Crew:
        '''Build a fresh crew with the current risk_strategy injected into risk_agent backstory.'''

        s_agent = Agent(
            role="Clinical Symptom Extraction Specialist",
            goal="Extract all clinically relevant data from the patient report in a structured format.",
            backstory="You extract facts. You do not diagnose. Format: 6 structured sections.",
            llm=llm, verbose=False
        )

        # ← Risk agent backstory updated each iteration from self.risk_strategy
        r_agent = Agent(
            role="Clinical Pharmacist — Drug Interaction Analyst",
            goal="Identify all drug interactions. Rate severity. Flag SEVERE/CONTRAINDICATED first.",
            backstory=self.risk_strategy,     # ← this is the POLICY being updated
            llm=llm, verbose=False
        )

        g_agent = Agent(
            role="Clinical Governance Validator",
            goal="Validate all outputs. Assign quality_score 1-10. Output: STATUS + quality_score + reasoning.",
            backstory=(
                "You score clinical AI pipeline outputs. "
                "You check: completeness of extraction, accuracy of interaction analysis, "
                "appropriateness of flagging. "
                "Always include 'quality_score: [n]' in your output."
            ),
            llm=llm, verbose=False
        )

        crew = Crew(
            agents=[s_agent, r_agent, g_agent],
            tasks=[
                Task(
                    description="Extract all data from: {patient_report}. Use 6 sections.",
                    expected_output="Structured 6-section extraction",
                    agent=s_agent
                ),
                Task(
                    description="Analyse all drug interactions for: {patient_report}. Flag SEVERE first.",
                    expected_output="Interaction table with severity ratings",
                    agent=r_agent
                ),
                Task(
                    description=(
                        "Review the extraction and interaction analysis for: {patient_report}.\n"
                        "Output: STATUS (APPROVED/FLAGGED/ESCALATE) + quality_score: [1-10] + 2-sentence reasoning."
                    ),
                    expected_output="STATUS + quality_score: [n] + 2-sentence reasoning",
                    agent=g_agent
                ),
            ],
            verbose=False
        )
        return crew

    def _extract_score(self, output: str) -> int:
        '''Parse quality_score from governance agent output text.'''
        # Match patterns like 'quality_score: 8' or 'quality_score:8'
        match = re.search(r'quality[_\s]?score[:\s]+([0-9]+)', output, re.IGNORECASE)
        if match:
            return min(10, max(1, int(match.group(1))))
        # Fallback inference from STATUS
        upper = output.upper()
        if "APPROVED" in upper:  return 8
        if "FLAGGED" in upper:   return 5
        if "ESCALATE" in upper:  return 3
        return 5

    def _update_strategy(self, score: int, governance_output: str) -> str:
        '''ADAPT step: if score < 7, rewrite risk_strategy using LLM.'''
        if score >= 7:
            return f"✓ Strategy maintained (score {score}/10 — above threshold)"

        improved = llm.invoke([
            SystemMessage(content=(
                "You improve clinical AI drug interaction analysis strategies. "
                "Output ONLY the improved strategy in 3 sentences. No headers."
            )),
            HumanMessage(content=(
                f"Current strategy: {self.risk_strategy}\n\n"
                f"Governance feedback: {governance_output[:400]}\n\n"
                "Write an improved drug interaction analysis strategy in 3 sentences:"
            )),
        ])

        old_strategy = self.risk_strategy[:70]
        self.risk_strategy = improved.content.strip()
        return (
            f"⟳ Strategy updated (score {score}/10)\n"
            f"  WAS: {old_strategy}...\n"
            f"  NOW: {self.risk_strategy[:80]}..."
        )

    def run_iteration(self, patient_report: str, iteration: int) -> int:
        '''Run one complete crew iteration + RL feedback.'''
        print(f"\n{'━'*58}")
        print(f"  ITERATION {iteration}")
        print(f"  Risk strategy: {self.risk_strategy[:65]}...")
        print(f"{'━'*58}")

        crew = self._build_crew()
        result = crew.kickoff(inputs={"patient_report": patient_report})
        result_text = str(result)

        # OBSERVE
        score = self._extract_score(result_text)
        self.learning_history.append({
            "iteration": iteration,
            "score":     score,
            "strategy":  self.risk_strategy[:80],
        })
        print(f"  Governance score: {score}/10")

        # ADAPT
        adaptation = self._update_strategy(score, result_text)
        print(f"  {adaptation[:110]}")
        return score

    def run_iterations(self, patient_report: str, n: int = 3):
        '''Run n iterations and display score trend.'''
        scores = []
        for i in range(1, n + 1):
            scores.append(self.run_iteration(patient_report, i))

        # Show score trend
        print(f"\n{'═'*58}")
        print("  SCORE TREND ACROSS ITERATIONS")
        print(f"{'═'*58}")
        for i, s in enumerate(scores, 1):
            bar = "█" * s + "░" * (10 - s)
            trend = "↑" if i > 1 and scores[i-1] > scores[i-2] else ("↓" if i > 1 and scores[i-1] < scores[i-2] else "→")
            print(f"  Iteration {i}: [{bar}] {s:2d}/10  {trend}")

        improving = scores[-1] >= scores[0]
        print(f"\n  Final strategy: {self.risk_strategy[:100]}...")
        print(f"\n  Overall: {'📈 Score IMPROVED' if improving else '📉 Score did not improve — debug the reward signal or strategy prompt'}")

print("ClinicalPipelineWithRL class ready. ✓")


In [ ]:
# ── Run 3 iterations on the most complex patient ─────────────
pipeline_rl = ClinicalPipelineWithRL()

cardiac_report = (
    "David Okafor, 52M. Post-MI 6 weeks. "
    "Medications: Clopidogrel 75mg, Atorvastatin 40mg, Aspirin 75mg. "
    "New: Clarithromycin 500mg BD (GP-prescribed for pneumonia). "
    "Symptoms: chest tightness, productive cough, temp 38.3°C. Allergies: None."
)

print("Running 3 reinforcement iterations on the cardiac patient.")
print("Clarithromycin + Clopidogrel interaction is the test case.\n")

pipeline_rl.run_iterations(patient_report=cardiac_report, n=3)


---
# Summary — Everything in One Table

| Part | Concept | Key class / object | Planning layer |
|------|---------|-------------------|---------------|
| 1 | Goal Execution Pipeline | `GoalExecutionPipeline` | Foundation |
| 2 | Hierarchical Planning | 5-agent `vacation_crew` | Layer 1 |
| 3 | Task Chaining | 4-agent `support_crew` + `expected_output` | Layer 1 + 2 |
| 4 | All 4 planning modes | `triage_crew` | All layers |
| 5 | Reinforcement Loop | `ReinforcementAgent.act/observe/adapt()` | Layer 3 |
| 6 | Hybrid ReAct | `hybrid_crew` — 3-layer architecture | All layers |
| 7 | RL + Crew integration | `ClinicalPipelineWithRL` | Full system |

---

## The 5 Laws of Production AI

Before deploying any agent system, verify:

- [ ] **Clear Goals** — every agent's `goal` is specific and measurable
- [ ] **Hierarchy** — tasks are decomposed; nothing is solved in one prompt
- [ ] **Reactive Layer** — agents read context, not hardcoded answers
- [ ] **Feedback Loop** — system has a mechanism to improve over time
- [ ] **Governance** — a monitoring/governance agent validates before action

---

*Week 11 — Professional Certificate Programme in Agentic AI and Applications*
